In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, md5, concat_ws, regexp_replace
from delta.tables import DeltaTable

def process_silver_scd2(resource_name, primary_key, attribute_cols):
    # UPDATED SCHEMA NOTATIONS
    bronze_table_name = f"bronze.{resource_name.lower()}"
    silver_table_name = f"silver.{resource_name.lower()}"

    print(f"Processing Silver layer for {resource_name}...")

    df_bronze = spark.read.table(bronze_table_name)

    # Clean patient_id at source using regexp_replace
    if resource_name == "Patient":
        df_staged = df_bronze.select(
            col("id").alias(primary_key),
            col("gender"),
            col("birthDate"),
            col("extraction_timestamp")
        ).filter(col(primary_key).isNotNull())
    elif resource_name == "Encounter":
        df_staged = df_bronze.select(
            col("id").alias(primary_key),
            col("status"),
            regexp_replace(col("subject.reference"), "^Patient/", "").alias("patient_id"),
            col("extraction_timestamp")
        ).filter(col(primary_key).isNotNull())
    elif resource_name == "Observation":
        df_staged = df_bronze.select(
            col("id").alias(primary_key),
            col("status"),
            regexp_replace(col("subject.reference"), "^Patient/", "").alias("patient_id"),
            col("extraction_timestamp")
        ).filter(col(primary_key).isNotNull())
    elif resource_name == "Condition":
        df_staged = df_bronze.select(
            col("id").alias(primary_key),
            col("clinicalStatus.coding")[0]["code"].alias("clinical_status"),
            regexp_replace(col("subject.reference"), "^Patient/", "").alias("patient_id"),
            col("extraction_timestamp")
        ).filter(col(primary_key).isNotNull())

    # Deduplicate staged records
    df_staged = df_staged.dropDuplicates([primary_key])

    # Add control columns & MD5 change hash
    hash_args = [col(c) for c in attribute_cols]
    df_staged = df_staged.withColumn("hash_key", md5(concat_ws("||", *hash_args))) \
                         .withColumn("effective_date", current_timestamp()) \
                         .withColumn("end_date", lit(None).cast("timestamp")) \
                         .withColumn("is_current", lit(True))

    # Initialize / Write Silver Delta Table
    if not spark.catalog.tableExists(silver_table_name):
        df_staged.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
        print(f"Created Silver table: {silver_table_name}\n" + "-"*50)
        return

    # SCD Type 2 Delta Merge
    target_table = DeltaTable.forName(spark, silver_table_name)

    target_table.alias("target").merge(
        df_staged.alias("source"),
        f"target.{primary_key} = source.{primary_key} AND target.is_current = true"
    ).whenMatchedUpdate(
        condition="target.hash_key != source.hash_key",
        set={
            "end_date": "source.effective_date",
            "is_current": "false"
        }
    ).execute()

    df_staged.write.format("delta").mode("append").saveAsTable(silver_table_name)
    print(f"Applied SCD Type 2 merge to {silver_table_name}\n" + "-"*50)

# Execute Silver load across all entities
process_silver_scd2("Patient", "patient_id", ["gender", "birthDate"])
process_silver_scd2("Encounter", "encounter_id", ["status", "patient_id"])
process_silver_scd2("Observation", "observation_id", ["status", "patient_id"])
process_silver_scd2("Condition", "condition_id", ["clinical_status", "patient_id"])

Processing Silver layer for Patient...
Created Silver table: silver.patient
--------------------------------------------------
Processing Silver layer for Encounter...
Created Silver table: silver.encounter
--------------------------------------------------
Processing Silver layer for Observation...
Created Silver table: silver.observation
--------------------------------------------------
Processing Silver layer for Condition...
Created Silver table: silver.condition
--------------------------------------------------
